# Python Syntax, Indentation, Comments, and Code Structure

**Topic:** `1.3` &nbsp;|&nbsp; **Module 1: Getting Started with Python**

*Why the left margin is part of the grammar: indentation as structure, the difference between comments and docstrings, and how to parse code without running it.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M1)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [01_getting_started/README.md](../README.md)

## 1. Why This Topic Matters

In every other language you learned, whitespace was decoration. In Python it is syntax, which means a formatting mistake is a parse error and a readable file is a correct file. Robotics teams also rely on structured, indented diagnostics, so this topic is where layout becomes data.

## 2. Learning Objectives

By the end of this topic you will be able to:

1. Explain how INDENT and DEDENT tokens produce block structure.
2. Apply PEP 8 indentation consistently and repair a mixed-indentation file.
3. Distinguish comments from docstrings and use each correctly.
4. Use implicit line joining to wrap long expressions readably.
5. Parse an indented log into nested data with an explicit stack.
6. Read docstrings from source with ast without importing the module.

## 3. Prerequisites

This topic assumes you already have:

- Topic 1.1 Introduction to Python
- Topic 1.2 Installing Python, IDEs, and the REPL

## 4. Mental Model

**Mental model: indentation is the syntax tree, drawn.** Every other language keeps structure somewhere you must consult - braces, keywords, a separate declaration. Python draws the structure in the left margin, so the file you read *is* the parse tree you are about to execute. When in doubt, re-indent the selection and the intended structure appears.

| You write | The interpreter receives |
| --- | --- |
| Four spaces at the start of a line | An INDENT token |
| A line back at the outer level | A DEDENT token |
| `if ready:` | A header, a colon, then a suite |
| `# note` | Nothing at all - it is discarded |
| `"""Doc"""` first in a function | A string expression bound to `__doc__` |

## 5. Terminology

Vocabulary used consistently throughout the course.

| Term | Meaning |
| --- | --- |
| Indentation | Leading whitespace that defines block structure in Python. |
| Suite | The set of statements belonging to one block header. |
| Header | The line ending in a colon that opens a block. |
| Block | A header plus its indented suite. |
| Comment | Text after `#`, discarded entirely by the parser. |
| Docstring | A string literal as the first statement of a module, class or function. |
| Implicit joining | A line break inside brackets, which the parser ignores. |
| Explicit joining | A trailing backslash, which continues the logical line. |
| PEP 8 | The Python style guide, which mandates four-space indentation. |
| TabError | Raised when tabs and spaces are mixed within one block. |

## 6. Conceptual Explanation

Most languages treat whitespace as insignificant: you may indent or not indent, and the meaning is carried by braces or keywords. Python inverts this. **Indentation is the grammar.** A suite of statements is defined by their common indentation, and the compiler rejects any file in which the indentation does not describe a consistent tree. The consequence is that a programmer reading your code sees the same structure the interpreter sees, with no chance of the two disagreeing.

The practical rule is simply four spaces per nesting level, consistently, everywhere. Python does not require four spaces - any consistent width parses - but four spaces is the PEP 8 convention, and mixing tabs with spaces is rejected outright because the width of a tab depends on the reader. Editors can therefore re-indent a file mechanically and produce the same structure the author intended.

In [ ]:
def report(battery_pct: float) -> str:
    if battery_pct < 20:
        verdict = 'LOW'
    elif battery_pct < 50:
        verdict = 'MEDIUM'
    else:
        verdict = 'OK'
    return verdict


for level in (10, 35, 90):
    print(f'{level:>3}% -> {report(level)}')

Comments are for the reader and are discarded entirely by the parser. A comment begins with `#` and runs to the end of the line. Docstrings are different: they are ordinary string expressions placed as the first statement of a module, class or function, which is why they can be read back at runtime with `__doc__`. Comments answer *why*; docstrings answer *what*.

In [ ]:
# A comment: explains why the threshold is what it is.
LOW_BATTERY_PCT = 20   # matches the fleet safety policy


def verdict(pct: float) -> str:
    """Return OK, MEDIUM or LOW for a battery percentage.

    This is a docstring: it is a real object attached to the function,
    retrievable at runtime and visible to help() and type checkers.
    """
    return 'LOW' if pct < LOW_BATTERY_PCT else 'OK'


print(verdict.__doc__.splitlines()[0])
print('docstring is a', type(verdict.__doc__).__name__)

> **Comments are not free**
>
> Because the parser discards them, a comment can drift out of date without any test failing. If code disagrees with its comment, delete the comment and write better code.

## 7. Formal Theory

The grammar describes an indented block as a **suite**: a sequence of simple statements followed by either a nested suite or a colon and nothing. Formally, a header line ends with a colon and its suite is everything indented further than the header. That is the whole rule, and every block form in the language - `if`, `for`, `while`, `def`, `class`, `with`, `match` - obeys it.

```text
header ':' NEWLINE INDENT statement+ DEDENT
```

1. The **lexer** produces INDENT and DEDENT tokens when the indentation of a line increases or decreases relative to the previous one.
2. The **parser** accepts a suite only after such a token, so inconsistent indentation is rejected before a single statement runs.
3. The compiler records the resulting nesting in a code object, which is why `inspect` and `dis` can report line numbers and nesting later.

In [ ]:
import dis


def verdict(pct):
    if pct < 20:
        return 'LOW'
    return 'OK'


code = verdict.__code__
print('first line :', code.co_firstlineno)
print('constants  :', code.co_consts)

| Construct | Header ends with | Suite is |
| --- | --- | --- |
| `if` / `elif` / `else` | colon | the indented block that follows |
| `for` / `while` | colon | the indented block that follows |
| `def` / `class` | colon | the indented block that follows |
| `with` | colon | the indented block that follows |
| Simple statement | no colon | the single statement on the line |

## 8. Syntax

The surface syntax is small on purpose. One statement per line, four-space indentation, `#` for comments, and a colon to open a block.

In [ ]:
# canonical structure
LOW_BATTERY_PCT = 20


def verdict(pct: float) -> str:
    """Return a battery verdict."""
    if pct < LOW_BATTERY_PCT:
        return 'LOW'
    return 'OK'

The anti-pattern - legal Python, but it fights the language:

```text
pct = 10; verdict = 'LOW'      # packed onto one line with a semicolon
if pct < 20: print(verdict)    # inline body hides the block
	print('tab indent')            # a tab, not four spaces
```

> **Semicolons and one-liners**
>
> Every semicolon in real code is a line the reader must mentally split again. Use them in a REPL for brevity, not in a file.

## 9. Basic Examples

**Example 1 - continuing a statement across lines.**

In [ ]:
message = (
    'ROBO-X '
    'ready'
)

coordinates = (12.0,
               3.5)

print(message)
print(coordinates)

Inside brackets a line break is insignificant, so a long call or literal can be wrapped freely. Outside brackets you need an explicit backslash, which is almost always a sign the line should be restructured instead.

## 10. Intermediate Examples

Wrapping a long call chain so that each argument sits on its own line keeps diff small and readable. This is the formatting that black produces.

In [ ]:
def send(robot_id: str, channel: str, value: float, *, timeout: float = 1.0) -> str:
    return f'{robot_id}:{channel}={value}@{timeout}s'


print(send(
    'rover-01',
    'temperature',
    22.5,
    timeout=2.5,
))

> **Trailing commas**
>
> A trailing comma inside brackets lets you add one item per line and keeps the formatter from collapsing the call back onto a single long line.

## 11. Advanced Examples

Because comments and docstrings survive into the compiled code object as constants, the parser can be asked what a source file contains without executing it. That is how documentation generators and linters work.

In [ ]:
import ast


source = '''
def send(robot_id, channel):
    """Publish one reading."""
    return robot_id, channel
'''

tree = ast.parse(source)
func = tree.body[0]
print('name  :', func.name)
print('args  :', [a.arg for a in func.args.args])
print('doc   :', ast.get_docstring(func))

## 12. Code Walkthrough

Below is a complete utility: a parser for indented log blocks. It is the shape of code you would write if a robot emitted structured, indented diagnostics and you needed them as data.

In [ ]:
"""parse_blocks.py - turn an indented log into nested dictionaries."""
from __future__ import annotations

def _width(line: str) -> int:
    return len(line) - len(line.lstrip(' '))


def parse_blocks(text: str) -> list[dict]:
    """Parse '  ' indented log lines into a tree of dicts.

    Each leaf line becomes {'text': ...} keyed by indentation depth.
    Blank lines and '#' comments are ignored.
    """
    roots: list[dict] = []
    stack: list[tuple[int, dict]] = []
    for raw in text.splitlines():
        if not raw.strip() or raw.lstrip().startswith('#'):
            continue
        width = _width(raw)
        node: dict = {'text': raw.strip()}
        while stack and stack[-1][0] >= width:
            stack.pop()
        if stack:
            stack[-1][1].setdefault('children', []).append(node)
        else:
            roots.append(node)
        stack.append((width, node))
    return roots

The algorithm is a single pass with an explicit stack. Each line pops every open level whose indentation is not shallower, then attaches the new node to whatever remains on top. That is the same idea the interpreter itself uses to turn INDENT and DEDENT tokens into nested blocks.

## 13. Line-by-Line Explanation

1. The docstring states the contract before any code, so a reader knows what the function returns without reading the body.
2. `_width` counts leading spaces only; `lstrip(' ')` with an explicit character avoids surprising behaviour on tab characters.
3. `roots` collects top-level nodes; `stack` holds (indent, node) pairs for the levels currently open.
4. Blank lines and comment lines are skipped *before* any width is computed, so a comment at column zero cannot corrupt the tree.
5. The `while stack and stack[-1][0] >= width` loop closes every level that is not shallower than the new line - this is the dedent operation.
6. Attaching to `stack[-1][1]` nests the node; otherwise it starts a new root.
7. Pushing the new node keeps the invariant that the stack mirrors the open blocks, which is what makes one pass sufficient.

## 14. Common Mistakes

**Mistake 1 - mixing tabs and spaces inside one block.**

```text
for i in range(2):
	if i:
            print(i)      # TabError / inconsistent use of tabs
```

**Mistake 2 - an inline body after a colon.**

```text
if ready: print('go')   # legal, but the block is invisible

if ready:
    print('go')       # the block is obvious at a glance
```

**Mistake 3 - a comment that contradicts the code.**

```text
# decrement the counter before use   <- left over from an older version
battery_pct -= 1
```

**Mistake 4 - using a docstring where a comment belongs.**

```text
def send(value):
    '''Calls the actuator.'''   # docstring: public API documentation
    # actuator.publish(value)    # comment: disabled for safety review
    return value
```

## 15. Debugging Techniques

Structure bugs are easy to locate because Python refuses to run the file. Read the traceback from the bottom: the final line names the problem, and the line number is where the parser gave up.

In [ ]:
import ast

broken = 'def f():\n    return 1\n   return 2\n'
try:
    ast.parse(broken)
except SyntaxError as exc:
    print('problem :', exc.msg)
    print('line     :', exc.lineno)
    print('offset   :', exc.offset)

Turning an unexpected `IndentationError` into an `unexpected indent` is almost always a matter of deleting a block header or un-indenting a line by hand. Editors can re-indent a whole selection mechanically, which is safer than adjusting whitespace one line at a time.

> **Read-only checks**
>
> `python -m py_compile file.py` and `ast.parse(source)` both validate structure without executing anything, so they are safe to run against code you do not trust.

## 16. Best Practices

- Indent with four spaces; never mix tabs and spaces.
- One statement per line; reserve semicolons for the REPL.
- Put a colon at the end of a block header and nothing else.
- Use a docstring on every module, class and public function.
- Use `#` comments for the reason a decision was made, not for restating code.
- Wrap long lines at around 88 characters, one argument per line.
- Prefer implicit line joining inside brackets over explicit backslashes.
- Re-indent mechanically rather than by hand when fixing whitespace.

## 17. Performance Considerations

Indentation has no direct runtime cost: the parser discards whitespace after building the tree, so a deeply indented function is no slower than a flat one. What does cost is nesting *depth* in the algorithms you write. A recursive function that descends one level per frame is bounded by Python's recursion limit, which defaults to 1000, and a deeply nested data structure walked recursively will hit it. In this course you will meet recursion in Module 4 and iterative alternatives in Module 2, so it is worth forming the habit early: when a structure has an obvious depth, write a loop with an explicit stack, as `parse_blocks` does, rather than a recursive descent.

The other measurable cost is re-parsing. `ast.parse` on a large source file is not free, so a tool that parses the same file repeatedly should cache the tree rather than re-parsing. In practice this rarely matters because tools run once per invocation, but it is the right instinct for anything long-lived.

## 18. Pythonic Approaches

Splitting text: explicit loop versus the method that says what it means.

In [ ]:
text = 'alpha\nbeta\ngamma'

# before: manual scanning for the newline character
lines, current = [], ''
for ch in text:
    if ch == '\n':
        lines.append(current)
        current = ''
    else:
        current += ch
if current:
    lines.append(current)

# after: the intent, stated directly
print(text.splitlines())
print(lines == text.splitlines())

## 19. Robotics Connection

Robot diagnostics are almost always emitted as indented text so that a human can read them, and a supervisor process frequently needs the same information as data. The parser from the walkthrough is exactly that bridge, and it runs against the simulator's own status output in the exercise that follows.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / 'shared').is_dir()), None)
if root is not None:
    sys.path.insert(0, str(root))

from shared.robo_x_sim import SimulatedRobot  # noqa: E402

robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
state = robot.status()

log = (f"robot {state['name']}\n"
       f"    battery {state['battery_pct']:.1f}%\n"
       f"    position {state['position']}\n")
print(log)

## 20. Engineering Example

A small formatter used by every command-line tool in the course: it rejects mixed indentation and reports the offending lines rather than guessing.

In [ ]:
def check_indentation(source: str, width: int = 4) -> list[str]:
    problems: list[str] = []
    for number, line in enumerate(source.splitlines(), start=1):
        if not line.strip():
            continue
        if '\t' in line[:len(line) - len(line.lstrip())]:
            problems.append(f'line {number}: tab in indentation')
            continue
        lead = len(line) - len(line.lstrip(' '))
        if lead % width:
            problems.append(f'line {number}: indent {lead} is not a multiple of {width}')
    return problems


sample = 'def f():\n    return 1\n'
print(check_indentation(sample) or 'indentation OK')

## 21. Guided Practice

1. Write a four-line function with a docstring and a nested `if` block.
2. Use `ast.parse` on your own function and print the docstring back.
3. Deliberately break the indentation and read the SyntaxError message.
4. Write `strip_comments(line)` that removes a trailing `#` comment and returns the code part, stripping trailing whitespace.

In [ ]:
import ast


def strip_comments(line: str) -> str:
    """Remove a trailing '#' comment from a line of code."""
    code = line.split('#', 1)[0]
    return code.rstrip()


print(repr(strip_comments('x = 1  # set the reading  # not a string')))
tree = ast.parse('def f():\n    """Doc."""\n    return 1\n')
print('docstring:', ast.get_docstring(tree.body[0]))

## 22. Knowledge Checks

Answer these before moving on. Full answers are in [solution.ipynb](solution.ipynb).

- **Q1.** What does the interpreter do with the suite that follows a colon-terminated header?
- **Q1.** What is the main engineering benefit of deriving block structure from indentation?
- **Q1.** Which change most reliably fixes a file that mixes tabs and spaces?
- **Q1.** Why might a fleet team parse a robot's indented status log into nested data?

## 23. Summary

Python's syntax is defined by layout. A header ending in a colon is followed by a suite of statements indented further than the header; the lexer emits INDENT and DEDENT tokens as the left margin moves, and the parser builds the tree from them. Because indentation is meaningful, the appearance of a program and its structure cannot disagree, which is the single most important consequence of the language's design.

Comments and docstrings occupy different roles and are implemented differently. A comment is discarded by the parser before anything is built, so it exists purely for a human reader and can rot silently. A docstring is an ordinary string expression placed first in a module, class or function; it survives into the compiled code object and is retrievable through `__doc__`, which is why tooling can read documentation without running your code.

For structure that is long or machine-oriented, prefer implicit line joining inside brackets and let the layout show the shape of the data. One statement per line keeps diffs honest. And when Python raises `IndentationError` or `TabError`, believe the parser: it has already proved your file's structure does not describe the tree you meant, and the fastest fix is usually a mechanical re-indent rather than an edit by hand.

## 24. Key Takeaways

- Indentation is grammar, not decoration.
- A block is a colon-terminated header plus a more-indented suite.
- Four spaces per level, consistently, never mixed with tabs.
- Comments are discarded; docstrings are real objects on `__doc__`.
- Inside brackets, line breaks are free; outside, avoid backslashes.
- One statement per line keeps diffs readable and review cheap.
- Structure bugs are parse errors, so they surface before any code runs.
- Use `ast.parse` to inspect code without executing it.

## 25. Further Exploration

- [PEP 8 - Style Guide](https://peps.python.org/pep-0008/) - indentation, line length and naming.
- [The Python reference: compound statements](https://docs.python.org/3/reference/compound_stmts.html)
- [tokenize - tokenize Python source](https://docs.python.org/3/library/tokenize.html) - shows INDENT and DEDENT directly.
- [ast - Abstract syntax trees](https://docs.python.org/3/library/ast.html)
- [black - the uncompromising formatter](https://black.readthedocs.io/) - settles layout arguments once.